# Kayak Project — Hotel Scraping

This notebook collects hotel data from Booking.com, for every destination that has weather data. The brief asks for hotels "for each french city", not only for the Top 5.

For each hotel:
- name
- score
- URL
- description (from the hotel page)
- coordinates (from the hotel page)

The next cell imports the libraries and sets the names of the files that keep track of the collection.

In [1]:
import os
import random
import time
from datetime import datetime
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

CARDS_FILE = "../data/raw/hotels_cards.csv"
SEARCH_LOG_FILE = "../data/raw/hotels_search_log.csv"
DETAILS_FILE = "../data/raw/hotels_details.csv"
PROGRESS_FILE = "../data/raw/collection_progress.log"

## Load the Destinations

Loads the clean weather file and takes the list of destinations to search. The output shows 34 cities.

In [2]:
city_weather = pd.read_csv("../data/clean/cities_weather.csv")
cities = city_weather["city"].tolist()
print("Cities to scrape:", len(cities))

Cities to scrape: 34


## Initialize Selenium Driver

Opens Chrome through Selenium. We need a real browser: Booking builds its pages with JavaScript, so a simple download would get a page without hotels. `CHROMEDRIVER_PATH` can point to a driver already on the computer; without it, the driver is downloaded. No output.

In [3]:
driver_path = os.getenv("CHROMEDRIVER_PATH") or ChromeDriverManager().install()
driver = webdriver.Chrome(service=Service(driver_path))

## Scrape Booking.com

Defines the tools of the collection. `scrape_booking` reads the name, the score and the URL of each hotel on a results page, through the `data-testid` attributes, which Booking keeps stable. `lang=fr` is added to every address, so that the score text is always in French. `check_not_blocked` stops the notebook at once if Booking shows a CAPTCHA or a block: we do not try to get around it. `pause` waits 5 to 9 seconds between pages. No output.

In [4]:
class BookingBlocked(Exception):
    pass


BLOCK_MARKERS = ("captcha", "verify you are human", "are you a robot", "access denied",
                 "unusual traffic", "vérifiez que vous êtes", "challenge-container", "awswaf")


def check_not_blocked(driver, expected_content):
    page = driver.page_source
    if expected_content in page:
        return
    text = (driver.title + " " + driver.current_url + " " + page[:200000]).lower()
    found = [marker for marker in BLOCK_MARKERS if marker in text]
    if found:
        raise BookingBlocked(f"{found} on {driver.title[:80]}")


def pause():
    time.sleep(random.uniform(5, 9))


def log_progress(message):
    with open(PROGRESS_FILE, "a", encoding="utf-8") as log:
        log.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} | {message}\n")


def in_french(url):
    parts = urlsplit(url)
    query = dict(parse_qsl(parts.query))
    query["lang"] = "fr"
    return urlunsplit(parts._replace(query=urlencode(query)))


def append_rows(rows, path):
    if rows:
        pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def scrape_booking(city):

    url = in_french("https://www.booking.com/searchresults.html?" + urlencode({"ss": f"{city}, France"}))

    driver.get(url)
    time.sleep(5)
    check_not_blocked(driver, 'data-testid="property-card"')

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select('[data-testid="property-card"]')

    hotels = []

    for card in cards:

        name = card.select_one('[data-testid="title"]')
        score = card.select_one('[data-testid="review-score"]')
        link = card.select_one("a")

        if link is None:
            continue

        hotels.append({
            "city": city,
            "hotel_name": name.text.strip() if name else None,
            "score": score.text.strip() if score else None,
            "url": in_french(link["href"]),
        })

    return hotels

## Collect Hotel Data

Searches each city and saves its hotels as soon as the page is read, so that a stop loses nothing. If the notebook runs again, it skips the cities already searched. It takes every hotel of the first results page: 15 per city, 16 for St Malo. The output shows 511 hotels for 34 cities. The brief does not set a number.

In [5]:
searched = set(pd.read_csv(SEARCH_LOG_FILE)["city"]) if os.path.exists(SEARCH_LOG_FILE) else set()

for city in cities:

    if city in searched:
        continue

    hotels = scrape_booking(city)
    append_rows(hotels, CARDS_FILE)
    append_rows([{"city": city, "n_cards": len(hotels)}], SEARCH_LOG_FILE)
    log_progress(f"search | {city} | {len(hotels)} hotels listed")
    print(city, len(hotels))
    pause()

cards_df = pd.read_csv(CARDS_FILE).drop_duplicates(subset=["city", "url"])
print("Cities searched:", pd.read_csv(SEARCH_LOG_FILE)["city"].nunique(), "| hotels listed:", len(cards_df))

Bormes les Mimosas 15


Bayonne 15


Biarritz 15


Cassis 15


Collioure 15


Marseille 15


... (25 outputs left out)

Colmar 15


Eguisheim 15


Chateau du Haut Koenigsbourg 15


Cities searched: 34 | hotels listed: 511


## Read Each Hotel Page

Defines the function that reads the coordinates and the description on a hotel page, in one visit. The results page only gives an address and a distance to the center, so the real description has to come from the hotel page. No output.

In [6]:
def get_hotel_details(driver, url):

    driver.get(url)
    time.sleep(3)
    check_not_blocked(driver, "data-atlas-latlng")

    try:
        map_element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-atlas-latlng]"))
        )
    except TimeoutException:
        return None, None, None

    lat, lon = map_element.get_attribute("data-atlas-latlng").split(",")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    description = soup.select_one('[data-testid="property-description"]')

    return float(lat), float(lon), description.get_text(" ", strip=True) if description else None

## Collect Coordinates and Descriptions

Visits each of the 511 hotel pages and saves each hotel at once. If the notebook runs again, it continues after the last hotel collected. It stops after 5 pages in a row without coordinates, because that may mean a block. The output shows 0 hotels already collected and 511 to collect at the start, then the end of the collection. The whole notebook took about 1 h 55.

In [7]:
done = set()
if os.path.exists(DETAILS_FILE):
    previous = pd.read_csv(DETAILS_FILE)
    done = set(zip(previous["city"], previous["url"]))
print("Already collected:", len(done), "| to collect:", len(cards_df) - len(done))

failures_in_a_row = 0

for _, row in cards_df.iterrows():

    if (row["city"], row["url"]) in done:
        continue

    lat, lon, description = get_hotel_details(driver, row["url"])

    append_rows([{**row.to_dict(), "description": description, "latitude": lat, "longitude": lon}], DETAILS_FILE)

    done.add((row["city"], row["url"]))
    log_progress(f"hotel page | {row['city']} | {len(done)}/{len(cards_df)} hotels collected")

    failures_in_a_row = failures_in_a_row + 1 if lat is None else 0
    if failures_in_a_row == 5:
        raise BookingBlocked("5 hotel pages in a row without coordinates")

    pause()

driver.quit()
print("Hotel pages collected ✅")

Already collected: 0 | to collect: 511


Hotel pages collected ✅


## Clean Hotel Scores

Turns the score text into a number. Booking writes the score twice, as in "note de 9,79,7", so the expression `note de (10|\d,\d)` keeps only the first value. The output shows 511 hotels, 1 without a score on Booking, no score text left unread, and no hotel without coordinates or description. The file is saved to `data/clean/hotels.csv`.

In [8]:
hotels_df = pd.read_csv(DETAILS_FILE)

hotels_df["score_clean"] = (
    hotels_df["score"]
    .astype(str)
    .str.extract(r'note de (10|\d,\d)')[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

unread = hotels_df["score"].notna() & hotels_df["score_clean"].isna()
print("Hotels:", hotels_df.shape)
print("Without a score on Booking:", hotels_df["score"].isna().sum(), "| score text not read:", unread.sum())
print("Without coordinates:", hotels_df["latitude"].isna().sum(), "| without description:", hotels_df["description"].isna().sum())

hotels_df.to_csv("../data/clean/hotels.csv", index=False)
print("Saved cleaned hotels file to ../data/clean/hotels.csv")

Hotels: (511, 8)
Without a score on Booking: 1 | score text not read: 0
Without coordinates: 0 | without description: 0
Saved cleaned hotels file to ../data/clean/hotels.csv


## Distance Between Hotels and Their City

For each city: the number of hotels, and the median and maximum distance to the geocoded point. A Booking search can return hotels far from the city, so we check it. No hotel is removed: the brief talks about hotels "in the area". Two cities stand out: Ariege (median 23.5 km, maximum 45.1 km), which is a department, and St Malo (median 20.6 km, maximum 26 km), where Booking mostly returns the surrounding area. No hotel is more than 50 km away.

In [9]:
def distance_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


with_city = hotels_df.merge(city_weather[["city", "latitude", "longitude"]], on="city", suffixes=("", "_city"))
with_city["distance_km"] = distance_km(with_city["latitude"], with_city["longitude"], with_city["latitude_city"], with_city["longitude_city"])

distance_report = (
    with_city.groupby("city")
    .agg(hotels=("hotel_name", "size"), median_km=("distance_km", "median"), max_km=("distance_km", "max"))
    .reindex(cities)
    .fillna({"hotels": 0})
    .sort_values("max_km", ascending=False)
)
distance_report.to_csv("../data/raw/hotels_distance_report.csv")
distance_report.round(1)

,hotels,median_km,max_km
city,,,
Ariege,15,23.5,45.1
St Malo,16,20.6,26.0
Chateau du Haut Koenigsbourg,15,3.8,11.1
Aix en Provence,15,0.8,9.3
Paris,15,3.2,6.3
Saintes Maries de la mer,15,0.6,5.6
Montauban,15,2.5,5.6
Nimes,15,0.8,5.5
Bayeux,15,0.5,4.8
